In [ ]:
import os

os.getcwd()

In [ ]:
from configs.columns import SystemColumns, ProcessColumns
from pipelines.pipeline_utils import extract_x_y
import pandas as pd
from typing import Any

from pipelines.grid_search_pipeline_executor import GridSearchPipelineExecutor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.tree import ExtraTreeRegressor


# Models

In [ ]:
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet
)
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

simple_regressors_for_grid = {
    "LinearRegression": {"classifier": [LinearRegression()],
                         "classifier__fit_intercept": [True, False]},
    "Ridge": {"classifier": [Ridge()],
              "classifier__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    "Lasso": {"classifier": [Lasso()],
              "classifier__alpha": [0.001, 0.01, 0.1, 1.0]},
    "ElasticNet": {"classifier": [ElasticNet()],
                   "classifier__max_iter": [10000],
                   "classifier__alpha": [0.001, 0.01, 0.1, 1.0],
                   "classifier__l1_ratio": [0.1, 0.5, 0.9]},
    "SVR": {"classifier": [SVR()],
            "classifier__kernel": ["rbf", "linear"],
            "classifier__C": [0.1, 1, 10],
            "classifier__epsilon": [0.01, 0.1]},
    "MLPRegressor": {"classifier": [MLPRegressor()],
                     "classifier__max_iter": [2000],
                     "classifier__hidden_layer_sizes": [(50,), (100,), (50, 50)],
                     "classifier__activation": ["relu", "tanh"],
                     "classifier__alpha": [0.0001, 0.001]}
}

In [ ]:
GradientBoostingRegressorModel = {"classifier": [GradientBoostingRegressor()],
                                  "classifier__loss": ["squared_error", "huber"],
                                  'classifier__max_depth': [80, 110],
                                  'classifier__max_features': [3],
                                  'classifier__min_samples_leaf': [3, 5],
                                  'classifier__min_samples_split': [8, 12],
                                  'classifier__n_estimators': [100, 500, 1000]}

ExtraTreeRegressorModel = {"classifier": [ExtraTreeRegressor()],
                           # 'classifier__n_estimators': [10, 50, 100],
                           'classifier__criterion': ['squared_error', 'absolute_error'],
                           'classifier__max_depth': [2, 16, 50],
                           'classifier__min_samples_split': [2, 6],
                           'classifier__min_samples_leaf': [1, 2],
                           # 'oob_score': [True, False],
                           'classifier__max_features': ['sqrt']}
# ElasticNetModel = {"classifier": [ElasticNet()],
#                    "classifier__max_iter": [5, 50],
#                    "classifier__alpha": [0.001, 0.01, 0.1],
#                    "classifier__l1_ratio": np.arange(0.0, 1.0, 0.1)}

HistGradientBoostingRegressorModel = {"classifier": [HistGradientBoostingRegressor()],
                                      "classifier__loss": ["squared_error", "quantile"],
                                      "classifier__quantile": [0.5, 0.6, 0.7],
                                      "classifier__max_iter": [400, 600, 800],
                                      "classifier__l2_regularization": [0.1, 0.3, 1.0, 3.0],
                                      'classifier__max_depth': [3, 4, 5, 6, 8],  #range(5, 16, 2),
                                      'classifier__min_samples_leaf': [20, 50, 100, 200]}  #range(10, 100, 10)}
ExtraTreesRegressorModel = {"classifier": [ExtraTreesRegressor()],
                            "classifier__max_depth": [3, 5, 7, 12],
                            "classifier__min_samples_leaf": [3, 7],
                            "classifier__min_weight_fraction_leaf": [0.1, 0.5],
                            "classifier__max_features": ["sqrt"],
                            "classifier__max_leaf_nodes": [10, 60, 90]}

RandomForestRegressorModel = {"classifier": [RandomForestRegressor()],
                              'classifier__n_estimators': [50, 100, 500, 1000],
                              'classifier__max_features': ['sqrt'],
                              'classifier__max_depth': [5, 7, 15, 60],
                              'classifier__min_samples_split': [2, 5, 10],
                              'classifier__min_samples_leaf': [1, 4]}

In [ ]:
all_possible_models = {
    "GradientBoostingRegressorModel": GradientBoostingRegressorModel,
    "ExtraTreesRegressorModel": ExtraTreesRegressorModel,
    "ExtraTreeRegressorModel": ExtraTreeRegressorModel,
    "HistGradientBoostingRegressorModel": HistGradientBoostingRegressorModel,
    "RandomForestRegressorModel": RandomForestRegressorModel
}

In [ ]:
process_df_path = r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\basic - system based - process of interest\all_durations\system_process_df_new.csv"

system_only_df_path = r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\basic - system based - process of interest\all_durations\system_only_df_new.csv"

system_target = SystemColumns.ENERGY_USAGE_SYSTEM_COL
process_target = ProcessColumns.ENERGY_USAGE_PROCESS_COL

In [ ]:
print(process_df_path)

In [ ]:
from typing import Callable


def run_grid_search(target_col: str, dataset_path: str, possible_models: list[dict],
                    scoring_methods: dict[str, str | Callable]) -> dict[str, Any]:
    grid_search_pipeline = GridSearchPipelineExecutor(possible_models=possible_models, scoring_methods=scoring_methods)
    dataset = pd.read_csv(dataset_path, index_col=0)
    X, y = extract_x_y(dataset, target_column=target_col)
    best_model_per_metric = grid_search_pipeline.run_grid_search(X, y)
    return best_model_per_metric

In [ ]:
def print_best_models(best_model: dict[str, Any], model_name: str) -> str:
    res = f"\n\nGrid Search Results for Model {model_name}: \n{best_model}"
    print(res)
    return res

In [ ]:
def run_grid_search_on_all_models(target_col: str, dataset_path: str, model_options: dict[str, dict[str, Any]], scoring_methods: dict[str, str | Callable]) -> tuple[dict[str, dict[str, Any]], str]:
    best_model_per_type = {}
    full_results = ""
    for model_name, model_dict in model_options.copy().items():
        full_results += f"\n\n***** Starting Grid Search for Model {model_name}: *****\n"
        print(f"\n\n***** Starting Grid Search for Model {model_name}: *****\n")
        best_model_per_metric = run_grid_search(target_col, dataset_path, [model_dict], scoring_methods)
        res = print_best_models(best_model_per_metric, model_name)
        full_results += res
        best_model_per_type[model_name] = best_model_per_metric
        print(f"\n\n***** Finished Grid Search for Model {model_name}: *****\n")
        full_results += f"\n\n***** Finished Grid Search for Model {model_name}: *****\n"

    return best_model_per_type, full_results


# Additional methods for metrics and loss functions

In [ ]:
import numpy as np
from sklearn.metrics import make_scorer


def relative_rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mean_y = np.mean(y_true)

    # Avoid division by zero
    if mean_y == 0:
        return np.inf

    return rmse / mean_y


rrmse_scorer = make_scorer(relative_rmse, greater_is_better=False)

# Find best system energy model - Real Time

In [ ]:
system_scoring_methods = {
    "relative_rmse_scorer": rrmse_scorer,
    "neg_mean_squared_error": "neg_mean_squared_error",
    "neg_root_mean_squared_error": "neg_root_mean_squared_error"
}

In [ ]:
print(system_only_df_path)

In [ ]:
best_system_models, results_system_txt = run_grid_search_on_all_models(system_target, system_only_df_path,
                                                                       all_possible_models, system_scoring_methods)

In [ ]:
with open(f"finetune_system_10_min_batches_basic_dataset_all_durations.txt", "w") as f:
    f.write(results_system_txt)

# Find Best Process Energy Model - Real Time

In [ ]:
process_scoring_methods = {
    "relative_rmse_scorer": rrmse_scorer,
    "neg_mean_squared_error": "neg_mean_squared_error",
    "neg_root_mean_squared_error": "neg_root_mean_squared_error"
}

In [ ]:
process_df_path

In [ ]:
best_process_models_basic, results_process_basic_txt = run_grid_search_on_all_models(process_target, process_df_path,
                                                                                     all_possible_models,
                                                                                     process_scoring_methods)

In [ ]:
with open(f"finetune_process_10_min_batches_basic_dataset_all_durations_new.txt", "w") as f:
    f.write(results_process_basic_txt)

# Find Long Term Models

In [ ]:
new_system_df_path = r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\aggregated - system based - process of interest\system_only_df.csv"
new_process_df_path = r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\aggregated - system based - process of interest\system_process_df.csv"

# Find best system energy model - Long Term

In [ ]:
system_scoring_methods = {
    "relative_rmse_scorer": rrmse_scorer,
    "neg_mean_squared_error": "neg_mean_squared_error",
    "neg_root_mean_squared_error": "neg_root_mean_squared_error"
}

In [ ]:
best_system_models_long, results_system_txt_long = run_grid_search_on_all_models(system_target, new_system_df_path,
                                                                                 all_possible_models,
                                                                                 system_scoring_methods)

In [ ]:
with open(f"finetune_system_10_min_batches_aggregated_dataset_all_durations.txt", "w") as f:
    f.write(results_system_txt_long)

# Find best process energy model - Long Term

In [ ]:
process_scoring_methods = {
    "relative_rmse_scorer": rrmse_scorer,
    "neg_mean_squared_error": "neg_mean_squared_error",
    "neg_root_mean_squared_error": "neg_root_mean_squared_error"
}

In [ ]:
best_process_models_long, results_process_txt_long = run_grid_search_on_all_models(process_target, new_process_df_path,
                                                                                   all_possible_models,
                                                                                   process_scoring_methods)

In [ ]:
with open(f"finetune_process_10_min_batches_aggregated_dataset_all_durations.txt", "w") as f:
    f.write(results_process_txt_long)

# Process model with simple regressors - Real Time

In [ ]:
process_scoring_methods = {
    "relative_rmse_scorer": rrmse_scorer,
    "neg_mean_squared_error": "neg_mean_squared_error",
    "neg_root_mean_squared_error": "neg_root_mean_squared_error"
}

In [ ]:
best_process_models_real_simple, results_process_txt_real_simple = run_grid_search_on_all_models(process_target,
                                                                                                 process_df_path,
                                                                                                 simple_regressors_for_grid,
                                                                                                 process_scoring_methods)

In [ ]:
with open(f"finetune_process_10_min_batches_basic_dataset_all_durations_simple_models.txt", "w") as f:
    f.write(results_process_txt_real_simple)

In [ ]:
model = {"classifier": [HistGradientBoostingRegressor()],
                                      "classifier__loss": ["quantile"],
                                      "classifier__quantile": [0.7],
                                      "classifier__max_iter": [600],
                                      "classifier__l2_regularization": [1.0],
                                      'classifier__max_depth': [8]}

In [ ]:
process_scoring_methods = {
    "neg_root_mean_squared_error": "neg_root_mean_squared_error"
}

In [ ]:
process_single, process_single_text = run_grid_search_on_all_models(process_target,
                                                                                                 process_df_path,
                                                                    {"hist": model},
                                                                                                 process_scoring_methods)